# Bedrock Gateway RAG Implementation

This notebook demonstrates how to implement Retrieval-Augmented Generation (RAG) using the GenAI Bedrock Gateway. RAG combines the power of retrieval systems with generative AI to produce more accurate, contextually relevant, and fact-based outputs.

## Features Covered
- 📝 Document processing and chunking
- 🔍 Text embedding generation for semantic search
- 🧠 Vector store creation and similarity search
- 💬 Context-aware LLM responses with retrieved information
- 📊 Performance metrics and optimization

# Initialization and Configuration

In [ ]:
import json
import time
import re
from typing import Dict, List, Any, Tuple

import boto3
import requests
import numpy as np
import pandas as pd
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError
from sklearn.metrics.pairwise import cosine_similarity



import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment-specific .env file (e.g., .env.dev, .env.test)
ENVIRONMENT = os.getenv("ENVIRONMENT", "dev")
env_file = Path(__file__).parent / f".env.{ENVIRONMENT}" if "__file__" in dir() else Path(f".env.{ENVIRONMENT}")
load_dotenv(env_file)

# Configuration
MODELS = {
    "llm": "us.anthropic.claude-sonnet-4-20250514-v1:0",  # For text generation
    "embedding": "amazon.titan-embed-text-v1"            # For embedding generation
}
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
API_URL = os.getenv("GATEWAY_API_URL", "https://your-gateway-url.com")
SECRET_ID = os.getenv("GATEWAY_SECRET_ID", "bedrock-gateway-dev-oauth-credentials")

# Extract the secret value from Secret Manager

session = boto3.Session(profile_name=os.getenv("AWS_PROFILE"), region_name=AWS_REGION)
JSON_SECRET = json.loads(session.client('secretsmanager').get_secret_value(
    SecretId=SECRET_ID)['SecretString'])
CLIENT_ID = JSON_SECRET['client_id']
CLIENT_SECRET = JSON_SECRET['client_secret']
TOKEN_URL = JSON_SECRET['token_url']

## Token Generation

In [ ]:
def generate_token():
    """Generate access token for API authentication"""
    payload = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "client_credentials",
        "audience": "bedrockproxygateway",
        "scope": "bedrockproxygateway:invoke"
    }

    response = requests.post(TOKEN_URL, data=payload)
    response.raise_for_status()
    token = response.json()['access_token']
    return f"Bearer {token}"

def create_bedrock_client(token):
    """Create and configure Bedrock runtime client"""
    bedrock = boto3.client(
        'bedrock-runtime',
        region_name=AWS_REGION,
        endpoint_url=API_URL,
        config=Config(signature_version=UNSIGNED, retries={'max_attempts': 0}),
        verify=False
    )

    def add_api_token(request, **kwargs):
        request.headers.add_header('Authorization', token)
        return request
    
    bedrock.meta.events.register('before-sign.bedrock-runtime.*', add_api_token)
    return bedrock

## Bedrock Runtime Client Setup

In [ ]:
# Generate token
TOKEN = generate_token()
print(f"Token generated: {TOKEN[:20]}...")

# Create Bedrock client
bedrock = create_bedrock_client(TOKEN)
print("Bedrock client configured successfully")

# 1. Document Processing

In this section, we'll create and process sample documents for our RAG system.

In [ ]:
# Create sample documents for our knowledge base
sample_documents = [
    {
        "title": "Introduction to AWS Bedrock",
        "content": """AWS Bedrock is a fully managed service that offers a choice of high-performing foundation models (FMs) 
        from leading AI companies like AI21 Labs, Anthropic, Cohere, Meta, Stability AI, and Amazon through a single API. 
        It offers a broad set of capabilities to build and scale generative AI applications. 
        Bedrock simplifies FM experimentation and fine-tuning, and provides enterprise-grade security and privacy. 
        With Bedrock, you can start building with generative AI on AWS in minutes and seamlessly transition to production without managing any infrastructure."""
    },
    {
        "title": "AWS Bedrock Features",
        "content": """AWS Bedrock offers the following key features:
        1. Single API Access: Access various foundation models through one unified API, simplifying implementation.
        2. Model Customization: Use your data to fine-tune models for specific use cases and improve performance.
        3. Secure Private Models: Keep your data private with no model training on customer data.
        4. Seamless Integration: Connect with AWS services like Lambda, SageMaker, and S3 for end-to-end applications.
        5. Enterprise-Ready: Get provisioned throughput, model evaluation, and private custom models."""
    },
    {
        "title": "Retrieval-Augmented Generation",
        "content": """Retrieval-Augmented Generation (RAG) is an AI framework that enhances large language models (LLMs) 
        by incorporating external knowledge sources. In RAG, when a query is submitted, relevant information is retrieved 
        from a knowledge base and provided to the LLM as additional context. This approach combines the benefits of 
        retrieval-based systems (accuracy, up-to-date information) with generative models (fluent, coherent responses). 
        RAG helps reduce hallucinations by grounding responses in factual data and allows models to access information 
        beyond their training data, including proprietary or recent information."""
    },
    {
        "title": "RAG vs. Fine-tuning",
        "content": """RAG (Retrieval-Augmented Generation) and fine-tuning are complementary approaches for enhancing LLM performance:
        RAG advantages:
        - No model retraining required, just update the knowledge base
        - Can handle large amounts of information without model size increases
        - Provides source attribution and transparency
        - Lower computational cost than fine-tuning
        Fine-tuning advantages:
        - Deeply integrates knowledge into model weights
        - Can improve stylistic aspects and domain-specific language
        - Potentially faster inference without retrieval overhead
        Many production systems use both approaches together: fine-tuning for core capabilities and RAG for accessing specific knowledge."""
    },
    {
        "title": "Vector Databases for RAG",
        "content": """Vector databases are specialized database systems designed to store and efficiently query vector embeddings. 
        They are crucial for RAG implementations because they enable semantic search based on meaning rather than keywords. 
        Popular vector database options include:
        1. Amazon OpenSearch: Fully managed service with vector search capabilities
        2. Pinecone: Purpose-built vector database optimized for machine learning applications
        3. Weaviate: Open-source vector search engine with multimodal capabilities
        4. Chroma: Lightweight embedding database for RAG applications
        5. FAISS (Facebook AI Similarity Search): Library for efficient similarity search
        Vector databases support approximate nearest neighbor (ANN) algorithms that enable fast similarity search even with millions of vectors."""
    }
]

print(f"Created {len(sample_documents)} sample documents for RAG testing")

## Document Chunking

Splitting documents into smaller chunks is essential for effective retrieval. Chunks should be sized appropriately for the embedding model and retrieval system. We'll implement a simple regex-based approach to split text into sentences.

In [ ]:
def split_into_sentences(text):
    """
    Split text into sentences using regex patterns.
    This is a simple implementation that handles common sentence endings.
    """
    # Replace line breaks with spaces
    text = text.replace("\n", " ")
    # Split on sentence-ending punctuation followed by space or end of string
    # This handles periods, question marks, exclamation marks
    sentence_pattern = r'(?<=[.!?])\s+'
    sentences = re.split(sentence_pattern, text)
    # Further clean sentences and remove empty ones
    cleaned_sentences = []
    for s in sentences:
        s = s.strip()
        if s:
            # Make sure sentence ends with punctuation
            if not s[-1] in '.!?':
                s += '.'
            cleaned_sentences.append(s)
    return cleaned_sentences

def chunk_document(document: Dict[str, str], max_chunk_size: int = 512) -> List[Dict[str, str]]:
    """Split document into smaller chunks based on sentences"""
    title = document["title"]
    content = document["content"]
    # Split content into sentences using our custom function
    sentences = split_into_sentences(content)
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        # If adding this sentence would exceed max_chunk_size, save current chunk and start a new one
        if len(current_chunk) + len(sentence) > max_chunk_size and current_chunk:
            chunks.append({
                "title": title,
                "content": current_chunk.strip(),
                "source": title
            })
            current_chunk = ""
        current_chunk += " " + sentence
    # Add the last chunk if it's not empty
    if current_chunk:
        chunks.append({
            "title": title,
            "content": current_chunk.strip(),
            "source": title
        })
    return chunks

# Process all documents into chunks
all_chunks = []
for doc in sample_documents:
    chunks = chunk_document(doc)
    all_chunks.extend(chunks)

# Display the chunks
print(f"Created {len(all_chunks)} chunks from {len(sample_documents)} documents\n")
for i, chunk in enumerate(all_chunks[:3]):
    print(f"Chunk {i+1}:")
    print(f"Title: {chunk['title']}")
    print(f"Content: {chunk['content'][:100]}...")
    print()

if len(all_chunks) > 3:
    print(f"... and {len(all_chunks) - 3} more chunks")

# 2. Embedding Generation

In this section, we'll generate vector embeddings for each document chunk using the Amazon Titan Text Embeddings model.

In [ ]:
def generate_embedding(text: str, model_id: str = MODELS["embedding"]) -> Tuple[List[float], float]:
    """Generate embedding vector for a text using Bedrock embedding model"""
    # Prepare request for embedding model
    request = {
        "inputText": text
    }
    try:
        # Measure latency for embedding generation
        start_time = time.time()
        # Call Bedrock invoke model API for embeddings
        response = bedrock.invoke_model(
            modelId=model_id,
            body=json.dumps(request),
            contentType="application/json",
            accept="application/json"
        )
        # Parse response
        response_body = json.loads(response["body"].read())
        embeddings = response_body["embedding"]
        end_time = time.time()
        latency_ms = (end_time - start_time) * 1000
        return embeddings, latency_ms
    except (ClientError, Exception) as e:
        print(f"ERROR: Can't invoke '{model_id}'. Reason: {e}")
        return [], 0

# Generate embeddings for all chunks
print("Generating embeddings for all document chunks...")
total_latency = 0

for i, chunk in enumerate(all_chunks):
    print(f"Processing chunk {i+1}/{len(all_chunks)}...")
    embedding, latency = generate_embedding(chunk["content"])
    chunk["embedding"] = embedding
    total_latency += latency
    # Print information about the embedding
    if embedding:
        print(f"✅ Generated embedding with {len(embedding)} dimensions in {latency:.2f}ms")
    else:
        print("❌ Failed to generate embedding")

print(f"\nCompleted embedding generation for {len(all_chunks)} chunks")
print(f"Total latency: {total_latency:.2f}ms")
print(f"Average latency per chunk: {total_latency / len(all_chunks):.2f}ms")

# 3. Vector Store Implementation

In this section, we'll implement a simple in-memory vector store using cosine similarity for demonstration purposes. In production, you would use a dedicated vector database like Amazon OpenSearch.

In [ ]:
class SimpleVectorStore:
    """Simple in-memory vector store for demonstration purposes"""
    def __init__(self, documents=None):
        self.documents = documents or []
    def add_documents(self, documents):
        """Add documents to the vector store"""
        self.documents.extend(documents)
    def search(self, query_embedding, top_k=3):
        """Search for most similar documents based on embedding"""
        if not self.documents:
            return []
        # Extract document embeddings
        document_embeddings = np.array([doc["embedding"] for doc in self.documents])
        query_embedding_array = np.array(query_embedding).reshape(1, -1)
        # Calculate cosine similarities
        similarities = cosine_similarity(query_embedding_array, document_embeddings)[0]
        # Get indices of top k most similar documents
        top_indices = similarities.argsort()[-top_k:][::-1]
        # Return top k documents with similarity scores
        results = []
        for idx in top_indices:
            doc = self.documents[idx].copy()  # Create a copy to avoid modifying the original
            doc["similarity"] = float(similarities[idx])  # Add similarity score
            results.append(doc)
        return results

# Create vector store and add documents
vector_store = SimpleVectorStore()
vector_store.add_documents(all_chunks)

print(f"Created vector store with {len(vector_store.documents)} documents")

# 4. RAG Query Implementation

Now we'll implement the full RAG pipeline to answer user queries using retrieved context:

In [ ]:
def rag_query(query: str, top_k: int = 3) -> Tuple[str, Dict[str, Any]]:
    """Process a query using the RAG pipeline"""
    metrics = {}
    # 1. Generate embedding for the query
    start_time = time.time()
    query_embedding, embedding_latency = generate_embedding(query)
    metrics["embedding_latency_ms"] = embedding_latency
    # 2. Retrieve relevant documents
    retrieval_start = time.time()
    retrieved_docs = vector_store.search(query_embedding, top_k=top_k)
    retrieval_latency = (time.time() - retrieval_start) * 1000
    metrics["retrieval_latency_ms"] = retrieval_latency
    # 3. Format context for the LLM
    context = ""
    for i, doc in enumerate(retrieved_docs):
        context += f"Document {i+1}: {doc['title']}\n"
        context += f"{doc['content']}\n\n"
        metrics[f"similarity_score_{i+1}"] = doc["similarity"]
    # 4. Create prompt with context and query
    prompt = f"""You are a helpful AI assistant with expertise in AWS services, particularly Bedrock and generative AI. 
    Use ONLY the following context to answer the question. If the context doesn't contain 
    the information needed to answer the question, say "I don't have enough information to answer this question."
    Context:
    {context}
    Question: {query}
    Answer:"""
    # 5. Send to LLM for response generation
    llm_start = time.time()
    conversation = [
        {"role": "user", "content": [{"text": prompt}]}
    ]
    try:
        response = bedrock.converse(
            modelId=MODELS["llm"],
            messages=conversation,
            inferenceConfig={"maxTokens": 512, "temperature": 0.3}
        )
        # Extract response text
        answer = response["output"]["message"]["content"][0]["text"]
        # Add response latency to metrics
        llm_latency = (time.time() - llm_start) * 1000
        metrics["llm_latency_ms"] = llm_latency
        metrics["total_latency_ms"] = embedding_latency + retrieval_latency + llm_latency
        return answer, metrics
    except (ClientError, Exception) as e:
        print(f"ERROR: LLM query failed. Reason: {e}")
        metrics["error"] = str(e)
        return "Error generating response", metrics

# Test the RAG pipeline with a sample query
test_query = "What is AWS Bedrock and what are its key features?"

print(f"Processing query: '{test_query}'\n")
answer, metrics = rag_query(test_query)

print("=== RAG Response ===\n")
print(answer)
print("\n=== Performance Metrics ===\n")
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

# 5. Advanced RAG Features

Let's explore some advanced RAG techniques to improve result quality:

In [ ]:
def enhanced_rag_query(query: str, top_k: int = 3, similarity_threshold: float = 0.5) -> Tuple[str, Dict[str, Any]]:
    """Enhanced RAG query with threshold filtering and query rewriting"""
    metrics = {}
    # 1. Generate embedding for the query
    query_embedding, embedding_latency = generate_embedding(query)
    metrics["embedding_latency_ms"] = embedding_latency
    # 2. Retrieve relevant documents
    retrieval_start = time.time()
    retrieved_docs = vector_store.search(query_embedding, top_k=top_k)
    retrieval_latency = (time.time() - retrieval_start) * 1000
    metrics["retrieval_latency_ms"] = retrieval_latency
    # 3. Filter by similarity threshold
    filtered_docs = [doc for doc in retrieved_docs if doc["similarity"] >= similarity_threshold]
    metrics["docs_retrieved"] = len(retrieved_docs)
    metrics["docs_after_filtering"] = len(filtered_docs)
    # If no documents meet the threshold, respond accordingly
    if not filtered_docs:
        return "I don't have enough relevant information to answer this question accurately.", metrics
    # 4. Format context with relevance scores
    context = ""
    for i, doc in enumerate(filtered_docs):
        context += f"Document {i+1} [Relevance: {doc['similarity']:.2f}]: {doc['title']}\n"
        context += f"{doc['content']}\n\n"
        metrics[f"similarity_score_{i+1}"] = doc["similarity"]
    # 5. Create prompt with context, query, and instructions for source attribution
    prompt = f"""You are a helpful AI assistant with expertise in AWS services, particularly Bedrock and generative AI.
    Use ONLY the provided context to answer the question. If the context doesn't contain the information needed,
    say "I don't have enough information to answer this question."
    Important instructions:
    1. Include specific source attribution when answering (e.g., "According to Document 1...")
    2. Synthesize information from multiple sources if needed
    3. Maintain accuracy - do not add information that's not in the context
    4. Provide a comprehensive answer based on all relevant context
    5. If some documents are irrelevant, focus on the most relevant ones
    Context:
    {context}
    Question: {query}
    Answer:"""
    # 6. Send to LLM for response generation with thinking
    llm_start = time.time()
    conversation = [
        {"role": "user", "content": [{"text": prompt}]}
    ]
    try:
        # Use Bedrock Converse API with thinking enabled
        response = bedrock.converse(
            modelId=MODELS["llm"],
            messages=conversation,
            inferenceConfig={"maxTokens": 1024, "temperature": 0.3}
        )
        # Extract response text
        answer = response["output"]["message"]["content"][0]["text"]
        # Add response latency to metrics
        llm_latency = (time.time() - llm_start) * 1000
        metrics["llm_latency_ms"] = llm_latency
        metrics["total_latency_ms"] = embedding_latency + retrieval_latency + llm_latency
        return answer, metrics
    except (ClientError, Exception) as e:
        print(f"ERROR: LLM query failed. Reason: {e}")
        metrics["error"] = str(e)
        return "Error generating response", metrics

# Test the enhanced RAG pipeline with different queries
test_queries = [
    "Explain the difference between RAG and fine-tuning for LLMs.",
    "What vector databases are recommended for RAG implementations?",
    "How does AWS Bedrock help with generative AI applications?"
]

for query in test_queries:
    print(f"\n{'='*60}\n")
    print(f"Query: {query}\n")
    answer, metrics = enhanced_rag_query(query)
    print("Response:")
    print(answer)
    print("\nPerformance Metrics:")
    print(f"Total latency: {metrics.get('total_latency_ms', 0):.2f}ms")
    print(f"Documents retrieved: {metrics.get('docs_retrieved', 0)}")
    print(f"Documents after filtering: {metrics.get('docs_after_filtering', 0)}")
    # Print top similarity scores
    similarity_scores = []
    for i in range(1, 4):  # Check for up to 3 similarity scores
        score_key = f"similarity_score_{i}"
        if score_key in metrics:
            similarity_scores.append(f"{metrics[score_key]:.4f}")
    if similarity_scores:
        print(f"Top similarity scores: {', '.join(similarity_scores)}")

# 6. Evaluation and Analysis

Let's perform a simple evaluation of our RAG system using test queries:

In [ ]:
def evaluate_rag_system(test_queries: List[str]) -> pd.DataFrame:
    """Evaluate the RAG system on multiple test queries"""
    results = []
    for query in test_queries:
        print(f"Evaluating query: '{query}'")
        answer, metrics = enhanced_rag_query(query)
        # Store results
        result = {
            "query": query,
            "total_latency_ms": metrics.get("total_latency_ms", 0),
            "embedding_latency_ms": metrics.get("embedding_latency_ms", 0),
            "retrieval_latency_ms": metrics.get("retrieval_latency_ms", 0),
            "llm_latency_ms": metrics.get("llm_latency_ms", 0),
            "docs_retrieved": metrics.get("docs_retrieved", 0),
            "docs_after_filtering": metrics.get("docs_after_filtering", 0),
        }
        # Add top similarity score if available
        if "similarity_score_1" in metrics:
            result["top_similarity_score"] = metrics["similarity_score_1"]
        results.append(result)
    # Convert to DataFrame for analysis
    df = pd.DataFrame(results)
    # Calculate summary statistics
    print("\nSummary Statistics:")
    summary_stats = df[["total_latency_ms", "embedding_latency_ms", "retrieval_latency_ms", "llm_latency_ms"]].describe()
    print(summary_stats)
    return df

# Define test queries that cover various aspects of our knowledge base
evaluation_queries = [
    "What is AWS Bedrock?",
    "How does RAG work with LLMs?",
    "Compare RAG and fine-tuning approaches.",
    "What vector databases can I use for RAG?",
    # "What are the key features of AWS Bedrock?"
]

# Evaluate the RAG system
evaluation_results = evaluate_rag_system(evaluation_queries)

# Display results in a table
evaluation_results